In [4]:
# chargement des bibliotheques
import pandas as pd

PATH = "../data/raw/DataCoSupplyChainDataset.csv"

In [5]:
# Importation du dataset et verifications
df = pd.read_csv(PATH, encoding="latin-1")
df.columns = df.columns.str.strip()

print(df.shape)
print(df["Order Id"].nunique())
print(df["Shipping Mode"].value_counts())
print(df["Delivery Status"].value_counts(normalize=True))

(180519, 53)
65752
Shipping Mode
Standard Class    107752
Second Class       35216
First Class        27814
Same Day            9737
Name: count, dtype: int64
Delivery Status
Late delivery        0.548291
Advance shipping     0.230402
Shipping on time     0.178352
Shipping canceled    0.042954
Name: proportion, dtype: float64


- Le dataset contient 180 519 lignes et 53 colonnes, ce qui offre un volume de données suffisant pour l'analyse
-  Les 180 519 lignes correspondent à 65 752 commandes distinctes. Une commande peut donc être répartie sur plusieurs lignes
- Le mode Standard Class est le plus utilisé, tandis que Same Day reste le moins fréquent
- Plus de la moitié des lignes sont associées à des livraisons en retard. Ce résultat mérite une analyse plus approfondie des délais de livraison

In [6]:
# repartition des commandes par statut
print(df["Order Status"].value_counts())
print(df.groupby("Shipping Mode")["Days for shipment (scheduled)"].agg(["min", "max", "nunique"]))

Order Status
COMPLETE           59491
PENDING_PAYMENT    39832
PROCESSING         21902
PENDING            20227
CLOSED             19616
ON_HOLD             9804
SUSPECTED_FRAUD     4062
CANCELED            3692
PAYMENT_REVIEW      1893
Name: count, dtype: int64
                min  max  nunique
Shipping Mode                    
First Class       1    1        1
Same Day          0    0        1
Second Class      2    2        1
Standard Class    4    4        1


- Les commandes COMPLETE et PENDING_PAYMENT sont les plus nombreuses. Les différents statuts reflètent plusieurs étapes du processus de traitement des commandes
- Chaque mode d'expédition possède un délai prévu fixe dans ce dataset, ce qui facilite la comparaison avec les délais réellement observés.

In [ ]:
# verif coherence : une même commande a-t-elle un seul mode ou un seul statut de livraison
g = df.groupby("Order Id")[["Shipping Mode", "Delivery Status", "Days for shipping (real)"]].nunique()
print((g > 1).sum())

Shipping Mode               0
Delivery Status             0
Days for shipping (real)    0
dtype: int64


Aucune commande ne présente de variation sur le mode d'expédition, le statut de livraison ou le délai réel analysé. Les données sont cohérentes sur ces trois critères

In [9]:
# Le statut "Late delivery" correspond-il a réel > prévu
late = df["Days for shipping (real)"] > df["Days for shipment (scheduled)"]
print(pd.crosstab(df["Delivery Status"], late))

col_0              False  True 
Delivery Status                
Advance shipping   41592      0
Late delivery          0  98977
Shipping canceled   3331   4423
Shipping on time   32196      0


Le retard calculé correspond bien au statut Late delivery. Les commandes annulées présentent toutefois des situations variées et doivent être étudiées séparément

In [10]:
# Dates, valeurs manquantes, encodage
dt = pd.to_datetime(df["order date (DateOrders)"], format="%m/%d/%Y %H:%M", errors="coerce")
print(dt.min(), dt.max(), dt.isna().sum())
print(df.isna().mean().sort_values(ascending=False).head(8))
# Vwerifier l'encodage
print(df["Order City"].str.contains("Ã", na=False).sum())

2015-01-01 00:00:00 2018-01-31 23:38:00 0
Product Description              1.000000
Order Zipcode                    0.862397
Customer Lname                   0.000044
Customer Zipcode                 0.000017
Days for shipment (scheduled)    0.000000
Sales per customer               0.000000
Benefit per order                0.000000
Delivery Status                  0.000000
dtype: float64
0


- Les commandes couvrent la période du 1er janvier 2015 au 31 janvier 2018, et aucune date n'a échoué lors de la conversion
- La colonne Product Description est entièrement vide et Order Zipcode contient beaucoup de valeurs manquantes. Ces deux colonnes nécessitent un traitement particulier avant l'analyse
- Aucun nom de ville ne contient la séquence de caractères recherchée. Ceci prouve un premier contrôle positif que tous les textes sont correctement encodés